# Partitioned ethnic groups

GREG stores one row per polygon, so a group shows up several times. We want groups whose polygons carry exactly two country codes, both in Sub-Saharan Africa.

**Expected:** 8,969 polygons, 913 groups, 184 split across two countries, 51 of those inside SSA.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

import config
import sample

In [2]:
units, partitioned = sample.run()

GREG polygons: 8969, distinct groups: 913
split across two countries: 184
both countries in SSA: 51

candidate units: 102 (51 groups)

most common country pairs:
country_pair
CM/NI    5
ET/SU    4
KE/TZ    3
PU/SG    2
ET/KE    2
GV/LI    2
CD/CM    2
AO/WA    2
MZ/TZ    2
CD/SU    2


In [3]:
units.head(10)

,G1ID,FIPS_CNTRY,name,COW,n_polygons,unit_id
0,13,SF,Afrikaners (Boers),560,6,13_SF
1,13,WA,Afrikaners (Boers),565,7,13_WA
2,45,MZ,Angoni,541,1,45_MZ
3,45,TZ,Angoni,510,1,45_TZ
4,111,CG,Bakonjo,490,1,111_CG
5,111,UG,Bakonjo,500,1,111_UG
6,114,PU,Balante,404,1,114_PU
7,114,SG,Balante,433,1,114_SG
8,124,AO,Bambundu,540,1,124_AO
9,124,CG,Bambundu,490,1,124_CG


In [4]:
partitioned['country_pair'].value_counts().head(15)

country_pair
CM/NI    5
ET/SU    4
KE/TZ    3
PU/SG    2
ET/KE    2
GV/LI    2
CD/CM    2
AO/WA    2
MZ/TZ    2
CD/SU    2
AO/ZA    2
SF/WA    2
ML/UV    1
CM/CT    1
ZA/ZI    1
Name: count, dtype: int64

In [5]:
greg = sample.load_greg()

id_cols = ["G1ID", "G2ID", "G3ID"]
print(greg[id_cols].dtypes.to_string())
for c in id_cols:
    s = greg[c]
    print(f"{c}: min={s.min()}, na={s.isna().sum()}, zeros={(s == 0).sum()}")

GREG polygons: 8969, distinct groups: 913
G1ID    int32
G2ID    int32
G3ID    int32
G1ID: min=1, na=0, zeros=0
G2ID: min=0, na=0, zeros=7383
G3ID: min=0, na=0, zeros=8935


In [6]:
def ids(col):
    s = greg[col].dropna()
    return set(s[s > 0].astype(int))

g1, g2, g3 = ids("G1ID"), ids("G2ID"), ids("G3ID")
union = g1 | g2 | g3

print(f"distinct G1ID: {len(g1)}")
print(f"distinct G2ID: {len(g2)}")
print(f"distinct G3ID: {len(g3)}")
print(f"union of all three: {len(union)}   (Weidmann et al. report 929)")
print(f"only ever G2 or G3: {sorted(union - g1)}")

per_poly = greg[id_cols].apply(lambda r: (r.dropna() > 0).sum(), axis=1)
print()
print(per_poly.value_counts().sort_index().to_string())

cand = set(partitioned["G1ID"])
hit = greg[greg["G2ID"].isin(cand) | greg["G3ID"].isin(cand)]
print(f"\npolygons where an SSA candidate is a secondary group: {len(hit)}")
if len(hit):
    print(hit[["FIPS_CNTRY"] + id_cols].to_string())

distinct G1ID: 913
distinct G2ID: 293
distinct G3ID: 11
union of all three: 928   (Weidmann et al. report 929)
only ever G2 or G3: [161, 285, 379, 563, 564, 603, 621, 724, 835, 840, 914, 962, 1184, 1231, 1260]

1    7383
2    1552
3      34

polygons where an SSA candidate is a secondary group: 23
     FIPS_CNTRY  G1ID  G2ID  G3ID
112          AO  1180  1192     0
115          AO  1180  1192     0
2039         CM   125  1095     0
2045         CM  1095   224     0
2222         ET    36   324     0
2224         ET    36   387     0
2254         ET    36   387     0
2272         ET    58   324     0
3495         KE    21   387     0
3509         KE   121   801     0
4112         NI   376   503     0
4617         SF  1040    13     0
4619         SF  1222    13     0
4620         SF  1222    13     0
4625         SF  1222    13     0
4628         SF   445    13     0
4639         SF   166    13     0
4640         SF   166    13     0
4641         SF   166    13     0
4642         SF   166